In [10]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv(r"/kaggle/input/datasets/karansridhar/karan-s-data-viz-class-co2-dataset/co2_emissions.csv")
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [11]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


In [12]:
HL_COLOR = '#C0392B'
GREY     = '#EEEEEE'   # if too faint on your screen, try '#DDDDDD'

def highlight_line_chart(data, value_col, highlight, title, y_title, label_fmt):
    """Grey lines for every country, one highlighted line with a direct end label."""
    data = data.sort_values(['Country', 'Year'])
    hl = data[data['Country'] == highlight]
    last_year, end = hl['Year'].iloc[-1], hl[value_col].iloc[-1]

    fig = go.Figure()

    # 1) Grey context lines first, so the highlight is drawn on top
    for country, g in data.groupby('Country'):
        if country == highlight:
            continue
        fig.add_trace(go.Scatter(
            x=g['Year'], y=g[value_col], mode='lines',
            line=dict(color=GREY, width=1),
            hoverinfo='skip', showlegend=False))

    # 2) Highlighted country
    fig.add_trace(go.Scatter(
        x=hl['Year'], y=hl[value_col], mode='lines',
        line=dict(color=HL_COLOR, width=3.5),
        showlegend=False,
        hovertemplate='%{x}: %{y:,.2f}<extra></extra>'))

    # 3) Direct label at the end of the line (no legend)
    fig.add_annotation(
        x=last_year, y=end,
        text=f"<b>{highlight}</b><br>{label_fmt.format(end)}",
        showarrow=False, xanchor='left', xshift=8,
        font=dict(color=HL_COLOR, size=13))

    # 4) Clean styling: no borders, no gridlines, horizontal tick text
    fig.update_layout(
        title=dict(text=title, x=0, xanchor='left', font=dict(size=20)),
        template='simple_white',
        plot_bgcolor='white',
        showlegend=False,
        margin=dict(l=60, r=130, t=80, b=50),   # right margin leaves room for the label
        xaxis=dict(title=None, showgrid=False, showline=False, tickangle=0, dtick=4),
        yaxis=dict(title=y_title, showgrid=False, showline=False, tickformat=','),
        width=900, height=520)
    return fig


def change_summary(data, value_col):
    """First vs last value per country, to help choose a story."""
    return (data.sort_values(['Country', 'Year']).groupby('Country')[value_col]
            .agg(first='first', last='last')
            .assign(change=lambda d: d['last'] - d['first'],
                    change_pct=lambda d: (d['last'] / d['first'] - 1) * 100)
            .round(2)
            .sort_values('change_pct', ascending=False))


print(change_summary(df, 'CO2_per_capita'))

                first  last  change  change_pct
Country                                        
China            1.32  4.37    3.05      231.06
India            1.21  3.36    2.15      177.69
Indonesia        1.37  3.22    1.85      135.04
Mexico           1.18  1.93    0.75       63.56
South Korea      1.32  2.15    0.83       62.88
Brazil           1.26  1.98    0.72       57.14
Australia        1.26  1.73    0.47       37.30
South Africa     1.20  1.46    0.26       21.67
Russia           1.19  1.12   -0.07       -5.88
Canada           1.17  1.05   -0.12      -10.26
Japan            1.10  0.89   -0.21      -19.09
United States    1.32  1.02   -0.30      -22.73
France           1.34  0.72   -0.62      -46.27
Germany          1.25  0.48   -0.77      -61.60
United Kingdom   1.15  0.31   -0.84      -73.04


In [13]:
# Task 1: CO2 per capita, all 15 countries, one highlighted
HIGHLIGHT_1 = 'China'   # change to any country from the table above

hl = df[df['Country'] == HIGHLIGHT_1].sort_values('Year')
y0, y1 = hl['Year'].iloc[0], hl['Year'].iloc[-1]
v0, v1 = hl['CO2_per_capita'].iloc[0], hl['CO2_per_capita'].iloc[-1]
pct = (v1 / v0 - 1) * 100
direction = "rose" if pct > 0 else "fell"

title_1 = (f"{HIGHLIGHT_1}'s CO₂ per capita {direction} {abs(pct):.0f}% "
           f"between {y0} and {y1}, from {v0:.2f} to {v1:.2f} t per person")

fig1 = highlight_line_chart(
    df, 'CO2_per_capita', HIGHLIGHT_1, title_1,
    y_title='CO₂ per capita (t per person)', label_fmt='{:.2f} t')
fig1.show()

In [14]:
#task-2:

# Task 2: CO2 emissions (Mt), Asian countries only
asia = df[df['Region'] == 'Asia'].copy()
print("Asian countries:", asia['Country'].unique())
print()
print(change_summary(asia, 'CO2_Mt'))

Asian countries: ['China' 'India' 'Japan' 'South Korea' 'Indonesia']

              first     last  change  change_pct
Country                                         
China        3513.8  12409.5  8895.7      253.16
India         901.6   2661.1  1759.5      195.15
Indonesia     382.8    930.5   547.7      143.08
South Korea   438.1    789.5   351.4       80.21
Japan        1165.7    865.1  -300.6      -25.79


In [15]:
# Task 2: total CO2 emissions, Asian countries, one highlighted
HIGHLIGHT_2 = 'China'   # change to any Asian country from the table above

hl = asia[asia['Country'] == HIGHLIGHT_2].sort_values('Year')
y0, y1 = hl['Year'].iloc[0], hl['Year'].iloc[-1]
v0, v1 = hl['CO2_Mt'].iloc[0], hl['CO2_Mt'].iloc[-1]
pct = (v1 / v0 - 1) * 100
direction = "grew" if pct > 0 else "fell"

# Only claim "highest in Asia" if the data supports it
last_vals = asia[asia['Year'] == y1].set_index('Country')['CO2_Mt']
is_top = last_vals.idxmax() == HIGHLIGHT_2
tail = ", the highest in Asia" if is_top else ""

title_2 = f"{HIGHLIGHT_2}'s CO₂ emissions {direction} {abs(pct):.0f}% between {y0} and {y1}{tail}"

fig2 = highlight_line_chart(
    asia, 'CO2_Mt', HIGHLIGHT_2, title_2,
    y_title='CO₂ emissions (Mt)', label_fmt='{:,.0f} Mt')
fig2.show()